In [ ]:
import venv, os, subprocess, sys

VENV_DIR = os.path.join(os.getcwd(), "sls-mkt_rag_venv")
venv.create(VENV_DIR, with_pip=True)

PYTHON = os.path.join(VENV_DIR, "Scripts", "python.exe")
print(f"Venv Python: {PYTHON}")
print(f"Kernel Python: {sys.executable}")  # Will differ — that's expected

In [ ]:
import os, psycopg
from pgvector.psycopg import register_vector
from dotenv import load_dotenv
load_dotenv(r"C:\Git projects\sales_marketing_rag\.env")
 
DIM = int(os.environ["EMBED_DIM"])
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)
conn.execute("CREATE EXTENSION IF NOT EXISTS vector;")
register_vector(conn)
 
conn.execute(f"""
CREATE TABLE IF NOT EXISTS documents (
    id          BIGSERIAL PRIMARY KEY,
    source      TEXT NOT NULL,          -- e.g. 'Q1-2026 earnings call'
    doc_type    TEXT NOT NULL,          -- 'transcript' | '10k'
    created_at  TIMESTAMPTZ DEFAULT now()
);
 
CREATE TABLE IF NOT EXISTS chunks (
    id          BIGSERIAL PRIMARY KEY,
    document_id BIGINT REFERENCES documents(id) ON DELETE CASCADE,
    chunk_index INT NOT NULL,
    content     TEXT NOT NULL,
    token_count INT,
    embedding   vector({DIM}),
    tsv         tsvector GENERATED ALWAYS AS (to_tsvector('english', content)) STORED
);
""")
print("schema ready")


schema ready
